In [1]:
"""
PDB STRUCTURE QUALITY SCORING — THREE WEIGHTING HYPOTHESES
==========================================================

Reads PDB_Data and Scoring_Ranges from the source workbook, assigns an
individual quality score (High=3, Medium=2, Low=1) to each of the eight
parameters, calculates weighted scores under three hypotheses, ranks the
records, and writes a new workbook with one worksheet per hypothesis.

Before running:
1. Close the source workbook in Excel.
2. Confirm the INPUT_FILE path below.
3. If your PDB_Data column labels differ, edit DATA_COLUMN_ALIASES.
4. Run this script in Jupyter using %run, or paste it into a notebook cell.

Required range columns:
    Parameter
    High quality (lower)
    Medium quality
    Low quality (higher)

Boundary interpretation:
    - Every lower boundary is inclusive.
    - An upper boundary marked with "<" is exclusive.
    - An upper boundary without "<" is inclusive.
    - If a value matches overlapping ranges, the script reports an error
      rather than silently choosing a category.
    - Values outside all three ranges are left unscored and reported.

Missing values:
    A missing/non-numeric parameter receives no individual score.
    The final score is the weighted mean over scored parameters only.
    Weight Coverage (%) and Parameters Scored are included for transparency.
    Rankings are descending; tied scores receive the same minimum rank.
"""

'\nPDB STRUCTURE QUALITY SCORING — THREE WEIGHTING HYPOTHESES\n==========================================================\n\nReads PDB_Data and Scoring_Ranges from the source workbook, assigns an\nindividual quality score (High=3, Medium=2, Low=1) to each of the eight\nparameters, calculates weighted scores under three hypotheses, ranks the\nrecords, and writes a new workbook with one worksheet per hypothesis.\n\nBefore running:\n1. Close the source workbook in Excel.\n2. Confirm the INPUT_FILE path below.\n3. If your PDB_Data column labels differ, edit DATA_COLUMN_ALIASES.\n4. Run this script in Jupyter using %run, or paste it into a notebook cell.\n\nRequired range columns:\n    Parameter\n    High quality (lower)\n    Medium quality\n    Low quality (higher)\n\nBoundary interpretation:\n    - Every lower boundary is inclusive.\n    - An upper boundary marked with "<" is exclusive.\n    - An upper boundary without "<" is inclusive.\n    - If a value matches overlapping ranges, the sc

In [3]:
from pathlib import Path
from decimal import Decimal, InvalidOperation
import re
import shutil
import math
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

In [5]:
# 1. FILE LOCATIONS — EDIT ONLY IF YOUR FILES ARE IN DIFFERENT LOCATIONS
# =============================================================================

SOURCE_FILE = Path(
    r"C:\Users\ASUS\OneDrive\Documents\PDB_Automation_Task\Final_PDB_Data_Collection_ITK_scoring_ranges_corrected.xlsx"
)

# Work locally to reduce problems with OneDrive file locking/synchronization.
WORKING_FOLDER = Path(r"C:\Users\ASUS\Documents\PDB_Scoring_Work")
OUTPUT_FILE = WORKING_FOLDER / "PDB_Quality_Scoring_All_Hypotheses.xlsx"

# If copying from OneDrive fails, close Excel, make the file available offline
# in OneDrive, and rerun. The script does not bypass Windows file permissions.

In [7]:
# 2. PARAMETERS AND PDB_DATA COLUMN NAME ALIASES
# =============================================================================

# The first alias in each list is the preferred displayed name.
# Matching ignores case, spaces, punctuation, %, and common Unicode variants.
# Add your exact source-column header to the relevant list if necessary.

PARAMETER_ALIASES = {
    "Resolution (Å)": [
        "Resolution (Å)", "Resolution", "Resolution (A)", "Resolution (Angstrom)",
    ],
    "Residue Count Difference": [
        "Residue Count Difference", "Residue count difference",
        "Residue Count Diff", "Residue Difference", "Residue Count",
    ],
    "R-Value Difference": [
        "R-Value Difference", "R Value Difference", "R-value difference",
        "R-value diff", "R Value Diff",
    ],
    "Clashscore": [
        "Clashscore", "Clash Score",
    ],
    "Ramachandran Outliers (%)": [
        "Ramachandran Outliers (%)", "Ramachandran Outliers",
        "Ramachandran Outlier (%)", "Ramachandran Outliers Percent",
    ],
    "Sidechain Outliers (%)": [
        "Sidechain Outliers (%)", "Sidechain Outliers",
        "Side-chain Outliers (%)", "Sidechain Outliers Percent",
        "Rotamer Outliers (%)", "Rotamer Outliers",
    ],
    "RSRZ Outliers (%)": [
        "RSRZ Outliers (%)", "RSRZ Outliers", "RSRZ Outliers Percent",
    ],
    "Average B, all atoms": [
        "Average B, all atoms", "Average B-factor", "Average B Factor",
        "Average B", "Average B-factor (Å²)", "Average B-factor (A2)",
        "Average B (all atoms)", "Average B all atoms",
        "Average B, all atoms (Å²)", "Average B all atoms (A^2)",
    ],
}

PARAMETERS = list(PARAMETER_ALIASES.keys())

In [9]:
# 3. HYPOTHESIS WEIGHTAGES — percentages must total 100 for each hypothesis
# =============================================================================

HYPOTHESES = {
    "Hypothesis_1": {
        "Resolution (Å)": 25,
        "R-Value Difference": 25,
        "Residue Count Difference": 10,
        "Clashscore": 8,
        "Ramachandran Outliers (%)": 8,
        "Sidechain Outliers (%)": 8,
        "RSRZ Outliers (%)": 8,
        "Average B, all atoms": 8,
    },
    "Hypothesis_2": {
        "Resolution (Å)": 20,
        "Residue Count Difference": 10,
        "R-Value Difference": 15,
        "Clashscore": 15,
        "Ramachandran Outliers (%)": 12,
        "Sidechain Outliers (%)": 12,
        "RSRZ Outliers (%)": 8,
        "Average B, all atoms": 8,
    },
    "Hypothesis_3": {
        "Resolution (Å)": 25,
        "Residue Count Difference": 10,
        "R-Value Difference": 20,
        "Clashscore": 10,
        "Ramachandran Outliers (%)": 10,
        "Sidechain Outliers (%)": 10,
        "RSRZ Outliers (%)": 10,
        "Average B, all atoms": 5,
    },
}

QUALITY_POINTS = {"high": 3, "medium": 2, "low": 1}

In [11]:
# 4. GENERAL HELPERS
# =============================================================================

def norm(text):
    """Normalize a header/label for robust comparison."""
    if text is None:
        return ""
    text = str(text).strip().lower()
    text = text.replace("å", "a").replace("²", "2").replace("−", "-")
    return re.sub(r"[^a-z0-9]+", "", text)


def validate_weights():
    expected = set(PARAMETERS)
    for name, weights in HYPOTHESES.items():
        if set(weights) != expected:
            missing = sorted(expected - set(weights))
            extra = sorted(set(weights) - expected)
            raise ValueError(
                f"{name} parameter mismatch. Missing={missing}; extra={extra}"
            )
        total = sum(weights.values())
        if total != 100:
            raise ValueError(f"{name} weights total {total}%, not 100%.")


def find_sheet_name(workbook, expected):
    """Find a worksheet by normalized exact name, with limited tolerant matching."""
    expected_n = norm(expected)
    for name in workbook.sheetnames:
        if norm(name) == expected_n:
            return name
    # Permit harmless separators or a suffix such as a version marker.
    for name in workbook.sheetnames:
        if expected_n in norm(name):
            return name
    return None


def find_header_row(ws, required_headers, max_scan=40):
    """Find a row containing all required header labels, ignoring formatting."""
    required = {norm(x) for x in required_headers}
    for r in range(1, min(ws.max_row, max_scan) + 1):
        present = {
            norm(ws.cell(r, c).value)
            for c in range(1, ws.max_column + 1)
            if ws.cell(r, c).value is not None
        }
        if required.issubset(present):
            return r
    return None


def to_decimal(value):
    """Convert Excel/string numeric values to Decimal; return None if missing/invalid."""
    if value is None or isinstance(value, bool):
        return None
    if isinstance(value, (int, float)):
        if not math.isfinite(float(value)):
            return None
        return Decimal(str(value))

    text = str(value).strip().replace(",", "")
    if norm(text) in {"", "na", "n/a", "none", "null", "notavailable", "-", "--", "?"}:
        return None

    # Accept numeric cells represented with a trailing percent sign.
    if text.endswith("%"):
        text = text[:-1].strip()

    try:
        result = Decimal(text)
        return result if result.is_finite() else None
    except (InvalidOperation, ValueError, TypeError):
        return None


In [13]:
# 5. RANGE PARSING AND INDIVIDUAL PARAMETER SCORING
# =============================================================================

RANGE_RE = re.compile(
    r"^\s*(-?(?:\d+(?:\.\d*)?|\.\d+))\s*"
    r"(?:to|–|—|-)\s*(<\s*)?"
    r"(-?(?:\d+(?:\.\d*)?|\.\d+))\s*$",
    flags=re.IGNORECASE,
)


def parse_range(text):
    """
    Return (lower, upper, upper_inclusive) for e.g. '1.35 to < 1.97'.
    """
    if text is None:
        raise ValueError("A quality range cell is blank.")

    match = RANGE_RE.fullmatch(str(text).strip())
    if not match:
        raise ValueError(
            f"Cannot parse range {text!r}. Expected e.g. '1.35 to < 1.97'."
        )

    lower = Decimal(match.group(1))
    upper = Decimal(match.group(3))
    upper_inclusive = not bool(match.group(2))

    if lower > upper:
        raise ValueError(f"Range lower bound exceeds upper bound: {text!r}")

    return lower, upper, upper_inclusive


def in_range(value, bounds):
    lower, upper, upper_inclusive = bounds
    return value >= lower and (value <= upper if upper_inclusive else value < upper)


def resolve_parameter(label):
    """Map a Scoring_Ranges parameter label to one of the eight canonical names."""
    label_n = norm(label)
    for canonical, aliases in PARAMETER_ALIASES.items():
        if label_n == norm(canonical) or any(label_n == norm(a) for a in aliases):
            return canonical

    # Additional common variations for Average B and outlier columns.
    compact_aliases = {
        "averageballatoms": "Average B, all atoms",
        "averagebfactorallatoms": "Average B, all atoms",
        "averagebfactor": "Average B, all atoms",
        "averageb": "Average B, all atoms",
        "ramachandranoutlierspercent": "Ramachandran Outliers (%)",
        "sidechainoutlierspercent": "Sidechain Outliers (%)",
        "rsrzoutlierspercent": "RSRZ Outliers (%)",
        "rvaluediff": "R-Value Difference",
        "residuecountdiff": "Residue Count Difference",
    }
    return compact_aliases.get(label_n)


def read_scoring_ranges(workbook):
    sheet = find_sheet_name(workbook, "Scoring_Ranges")
    if sheet is None:
        raise ValueError(
            "Could not find the scoring-ranges worksheet. "
            f"Worksheets present: {workbook.sheetnames}. "
            "Rename the worksheet to 'Scoring_Ranges' or update find_sheet_name()."
        )

    ws = workbook[sheet]
    required = [
        "Parameter", "High quality (lower)", "Medium quality", "Low quality (higher)"
    ]
    header_row = find_header_row(ws, required)

    if header_row is None:
        raise ValueError(
            f"Could not find the required four headers in worksheet {sheet!r}. "
            "Check the header wording and layout."
        )

    cols = {}
    for c in range(1, ws.max_column + 1):
        value = ws.cell(header_row, c).value
        if value is not None:
            cols[norm(value)] = c

    parameter_col = cols[norm("Parameter")]
    high_col = cols[norm("High quality (lower)")]
    medium_col = cols[norm("Medium quality")]
    low_col = cols[norm("Low quality (higher)")]

    ranges = {}
    unrecognized_labels = []

    for r in range(header_row + 1, ws.max_row + 1):
        label = ws.cell(r, parameter_col).value
        if label is None or not str(label).strip():
            continue

        canonical = resolve_parameter(label)
        if canonical is None:
            unrecognized_labels.append(str(label).strip())
            continue

        if canonical in ranges:
            raise ValueError(
                f"Duplicate scoring-range rows map to {canonical!r}; "
                "please keep one row per parameter."
            )

        ranges[canonical] = {
            "high": parse_range(ws.cell(r, high_col).value),
            "medium": parse_range(ws.cell(r, medium_col).value),
            "low": parse_range(ws.cell(r, low_col).value),
            "source_label": str(label).strip(),
        }

    missing = [p for p in PARAMETERS if p not in ranges]
    if missing:
        raise ValueError(
            "Scoring ranges are missing for these required parameters:\n- "
            + "\n- ".join(missing)
            + (
                "\nUnrecognized labels encountered: " + ", ".join(unrecognized_labels)
                if unrecognized_labels else ""
            )
        )

    # Check that ranges do not overlap. Gaps are allowed and will remain unscored.
    for parameter, three in ranges.items():
        categories = [three["high"], three["medium"], three["low"]]
        for i in range(len(categories)):
            for j in range(i + 1, len(categories)):
                a, b = categories[i], categories[j]
                # Test overlap of numeric intervals.
                left = max(a[0], b[0])
                right = min(a[1], b[1])
                if left < right:
                    raise ValueError(
                        f"Overlapping ranges for {parameter}: {a} and {b}. "
                        "Please correct the Scoring_Ranges worksheet."
                    )
                if left == right and a[2] and b[2]:
                    raise ValueError(
                        f"Overlapping inclusive boundary for {parameter}: {a} and {b}."
                    )

    print(f"Scoring ranges loaded from worksheet: {sheet}")
    for p in PARAMETERS:
        print(f"  {p}: high={ranges[p]['high']}, medium={ranges[p]['medium']}, low={ranges[p]['low']}")
    return ranges


def score_value(value, parameter_ranges):
    number = to_decimal(value)
    if number is None:
        return None, "Missing/non-numeric"

    matched = [
        category for category in ("high", "medium", "low")
        if in_range(number, parameter_ranges[category])
    ]

    if len(matched) == 1:
        category = matched[0]
        return QUALITY_POINTS[category], category.title()

    if len(matched) > 1:
        raise ValueError(
            f"Value {value!r} matched multiple quality ranges: {matched}. "
            "Check the scoring boundaries."
        )

    return None, "Outside defined ranges"

In [15]:
# 6. PDB_DATA LOADING AND COLUMN RESOLUTION
# =============================================================================

def resolve_data_columns(df):
    normalized_to_actual = {}
    for col in df.columns:
        normalized_to_actual.setdefault(norm(col), []).append(col)

    mapping = {}
    missing = []

    for canonical, aliases in PARAMETER_ALIASES.items():
        candidates = [canonical] + aliases
        actual = None

        for candidate in candidates:
            matches = normalized_to_actual.get(norm(candidate), [])
            if len(matches) == 1:
                actual = matches[0]
                break
            if len(matches) > 1:
                raise ValueError(
                    f"More than one PDB_Data column matches {candidate!r}: {matches}. "
                    "Rename the duplicate columns so each parameter is unambiguous."
                )

        if actual is None:
            missing.append((canonical, aliases))
        else:
            mapping[canonical] = actual

    if missing:
        details = "\n".join(
            f"  - {canonical}: tried {aliases}"
            for canonical, aliases in missing
        )
        raise ValueError(
            "Could not map all eight parameters to columns in PDB_Data.\n"
            f"Actual columns: {list(df.columns)}\n"
            f"Unmatched parameters:\n{details}\n"
            "Add the exact source header to PARAMETER_ALIASES near the top of the script."
        )

    return mapping


def get_pdb_data_sheet(workbook):
    sheet = find_sheet_name(workbook, "PDB_Data")
    if sheet is None:
        raise ValueError(
            "Could not find the PDB_Data worksheet. "
            f"Worksheets present: {workbook.sheetnames}"
        )
    return sheet


def load_pdb_dataframe(input_file, sheet_name):
    # Read the sheet without assuming the header is in row 1. Find a row
    # containing a recognizable PDB identifier or parameter header.
    raw = pd.read_excel(input_file, sheet_name=sheet_name, header=None, engine="openpyxl")

    header_idx = None
    for i in range(min(len(raw), 40)):
        row_values = [norm(v) for v in raw.iloc[i].tolist() if pd.notna(v)]
        if (
            any(x in {"pdbid", "pdb", "pdbidentifier", "pdbentry"} for x in row_values)
            or sum(x in {norm(a) for aliases in PARAMETER_ALIASES.values() for a in aliases + [aliases[0]]}
                   for x in row_values) >= 4
        ):
            header_idx = i
            break

    if header_idx is None:
        # Standard Excel table: row 1 is header.
        header_idx = 0

    df = pd.read_excel(
        input_file, sheet_name=sheet_name, header=header_idx, engine="openpyxl"
    )

    # Remove completely empty rows and columns, but preserve records with partial values.
    df = df.dropna(axis=0, how="all").dropna(axis=1, how="all").reset_index(drop=True)
    df.columns = [str(c).strip() for c in df.columns]
    return df

In [17]:
# 7. SCORE A HYPOTHESIS
# =============================================================================

def score_hypothesis(df, column_map, ranges, weights, hypothesis_name):
    result = df.copy()

    # Individual category and numeric scores for every parameter.
    for parameter in PARAMETERS:
        source_col = column_map[parameter]
        scored = result[source_col].apply(
            lambda x, p=parameter: score_value(x, ranges[p])
        )
        result[f"{parameter} — Quality"] = scored.apply(lambda pair: pair[1])
        result[f"{parameter} — Score"] = scored.apply(lambda pair: pair[0])

        # Contribution is displayed on the 0–3 score scale, weighted as a fraction.
        result[f"{parameter} — Weighted Contribution"] = result[
            f"{parameter} — Score"
        ].apply(
            lambda score, w=weights[parameter]:
            float(score) * w / 100 if pd.notna(score) else None
        )

    final_scores = []
    coverage_values = []
    scored_counts = []

    for _, row in result.iterrows():
        weighted_sum = 0.0
        available_weight = 0
        count = 0

        for parameter in PARAMETERS:
            score = row[f"{parameter} — Score"]
            if pd.notna(score):
                w = weights[parameter]
                weighted_sum += float(score) * w
                available_weight += w
                count += 1

        # Normalize to available weight so the final remains on a 1–3 scale.
        final_scores.append(
            weighted_sum / available_weight if available_weight else None
        )
        coverage_values.append(available_weight)
        scored_counts.append(count)

    result["Final Score"] = final_scores
    result["Rank"] = result["Final Score"].rank(
        ascending=False, method="min"
    )
    result.loc[result["Final Score"].isna(), "Rank"] = None
    result["Parameters Scored"] = scored_counts
    result["Weight Coverage (%)"] = coverage_values

    # Add the hypothesis name for easy identification if sheets are exported/combined.
    result.insert(0, "Scoring Hypothesis", hypothesis_name)

    # Sort highest score first, while keeping unscored rows at the end.
    sort_cols = ["Final Score"]
    ascending = [False]
    possible_id_cols = ["PDB ID", "PDB_ID", "PDB", "PDB Entry", "PDB identifier"]
    id_col = next((c for c in possible_id_cols if c in result.columns), None)
    if id_col:
        sort_cols.append(id_col)
        ascending.append(True)

    result = result.sort_values(
        by=sort_cols, ascending=ascending, na_position="last", kind="stable"
    ).reset_index(drop=True)

    # Keep source columns first, followed by per-parameter outputs, then summary.
    score_cols = []
    for p in PARAMETERS:
        score_cols.extend([
            f"{p} — Quality",
            f"{p} — Score",
            f"{p} — Weighted Contribution",
        ])

    summary_cols = [
        "Final Score", "Rank", "Parameters Scored", "Weight Coverage (%)"
    ]
    front = ["Scoring Hypothesis"]
    source_cols = [c for c in df.columns if c not in front]
    ordered = front + source_cols + score_cols + summary_cols
    return result[[c for c in ordered if c in result.columns]]

In [19]:
# 8. OUTPUT FORMATTING AND METHODOLOGY
# =============================================================================

def write_dataframe(workbook, sheet_name, df):
    if sheet_name in workbook.sheetnames:
        del workbook[sheet_name]

    ws = workbook.create_sheet(sheet_name)
    ws.append(list(df.columns))

    for row in df.itertuples(index=False, name=None):
        values = []
        for value in row:
            if pd.isna(value):
                values.append(None)
            elif hasattr(value, "item"):
                values.append(value.item())
            else:
                values.append(value)
        ws.append(values)

    style_worksheet(ws)
    return ws


def style_worksheet(ws):
    header_fill = PatternFill(fill_type="solid", fgColor="305496")
    header_font = Font(bold=True, color="FFFFFF")

    for cell in ws[1]:
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(
            horizontal="center", vertical="center", wrap_text=True
        )

    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions
    ws.row_dimensions[1].height = 44

    for c in range(1, ws.max_column + 1):
        header = str(ws.cell(1, c).value or "")
        max_len = min(max(len(header), 12), 42)

        # Sample rows to estimate readable width.
        for r in range(2, min(ws.max_row, 250) + 1):
            val = ws.cell(r, c).value
            if val is not None:
                max_len = min(max(max_len, len(str(val)) + 1), 42)

        ws.column_dimensions[get_column_letter(c)].width = max_len

        if header == "Final Score" or "Weighted Contribution" in header:
            for r in range(2, ws.max_row + 1):
                ws.cell(r, c).number_format = "0.0000"
        elif header == "Rank":
            for r in range(2, ws.max_row + 1):
                ws.cell(r, c).number_format = "0"
        elif header == "Weight Coverage (%)":
            for r in range(2, ws.max_row + 1):
                ws.cell(r, c).number_format = '0"%"'

    summary_names = {
        "Final Score", "Rank", "Parameters Scored", "Weight Coverage (%)"
    }
    for c in range(1, ws.max_column + 1):
        if ws.cell(1, c).value in summary_names:
            for r in range(1, ws.max_row + 1):
                ws.cell(r, c).fill = PatternFill(fill_type="solid", fgColor="DDEBF7")


def create_method_sheet(workbook, ranges):
    name = "Scoring_Method"
    if name in workbook.sheetnames:
        del workbook[name]
    ws = workbook.create_sheet(name)

    rows = [
        ["Item", "Description"],
        ["Quality points", "High quality = 3; Medium quality = 2; Low quality = 1."],
        ["Ranges source", "Ranges are read from the Scoring_Ranges worksheet in the input workbook."],
        ["Boundary rules", "Lower endpoint inclusive. Upper endpoint is exclusive when written '<'; otherwise inclusive."],
        ["Outside range", "A value outside all defined ranges is not scored and is identified as 'Outside defined ranges'."],
        ["Missing data", "Blank or non-numeric values receive no individual score."],
        ["Final score", "Weighted mean of available parameter scores, normalized by the sum of weights for parameters that were scored; range is 1–3."],
        ["Weight coverage", "Sum of weight percentages for parameters successfully scored; maximum is 100%."],
        ["Rank", "Descending final score. Ties share the same minimum rank. Rows without a final score have no rank."],
        ["Hypothesis 1 weights", str(HYPOTHESES["Hypothesis_1"])],
        ["Hypothesis 2 weights", str(HYPOTHESES["Hypothesis_2"])],
        ["Hypothesis 3 weights", str(HYPOTHESES["Hypothesis_3"])],
        ["Loaded scoring ranges", ""],
    ]

    for row in rows:
        ws.append(row)

    for p in PARAMETERS:
        r = ranges[p]
        ws.append([
            p,
            f"High: {r['source_label']} -> {r['high']}; "
            f"Medium: {r['medium']}; Low: {r['low']}"
        ])

    style_worksheet(ws)
    ws.column_dimensions["A"].width = 32
    ws.column_dimensions["B"].width = 115


In [21]:
# 9. MAIN
# =============================================================================

def main():
    validate_weights()
    WORKING_FOLDER.mkdir(parents=True, exist_ok=True)

    if not SOURCE_FILE.exists():
        raise FileNotFoundError(
            f"Input workbook not found:\n{SOURCE_FILE}\n"
            "Verify the path and ensure the OneDrive file is downloaded locally."
        )

    # Copy the source file locally. If the destination exists, overwrite it.
    try:
        shutil.copy2(SOURCE_FILE, WORKING_FOLDER / SOURCE_FILE.name)
        local_input = WORKING_FOLDER / SOURCE_FILE.name
    except PermissionError as exc:
        raise PermissionError(
            "Windows denied access while copying the source workbook. "
            "Close it in Excel, ensure OneDrive has downloaded it, and retry."
        ) from exc

    print(f"Using input workbook: {local_input}")

    try:
        source_wb = load_workbook(local_input, data_only=True, read_only=True)
    except PermissionError as exc:
        raise PermissionError(
            "Windows denied access while reading the workbook. "
            "Close Excel and check file permissions."
        ) from exc

    pdb_sheet = get_pdb_data_sheet(source_wb)
    ranges = read_scoring_ranges(source_wb)
    source_wb.close()

    df = load_pdb_dataframe(local_input, pdb_sheet)
    if df.empty:
        raise ValueError(f"The {pdb_sheet!r} worksheet contains no data rows.")

    column_map = resolve_data_columns(df)

    print(f"\nPDB data worksheet: {pdb_sheet}")
    print(f"Rows to score: {len(df)}")
    print("Parameter-to-source-column mapping:")
    for p, col in column_map.items():
        print(f"  {p}  <-  {col}")

    # Load the workbook with formulas preserved so the original sheets remain.
    try:
        output_wb = load_workbook(local_input)
    except PermissionError as exc:
        raise PermissionError(
            "Cannot reopen workbook for writing. Close it in Excel and retry."
        ) from exc

    for hypothesis_name, weights in HYPOTHESES.items():
        print(f"\nScoring {hypothesis_name} ...")
        scored = score_hypothesis(
            df=df,
            column_map=column_map,
            ranges=ranges,
            weights=weights,
            hypothesis_name=hypothesis_name,
        )
        write_dataframe(output_wb, hypothesis_name, scored)

        valid = scored["Final Score"].notna().sum()
        print(f"  Finished. Records with a final score: {valid}/{len(scored)}")

    create_method_sheet(output_wb, ranges)

    try:
        output_wb.save(OUTPUT_FILE)
    except PermissionError as exc:
        raise PermissionError(
            f"Cannot save output workbook:\n{OUTPUT_FILE}\n"
            "Close the output file in Excel if it is already open, then rerun."
        ) from exc
    finally:
        output_wb.close()

    print("\n" + "=" * 72)
    print("SCORING FINISHED")
    print("=" * 72)
    print(f"Output workbook:\n{OUTPUT_FILE}")
    print("Worksheets: Hypothesis_1, Hypothesis_2, Hypothesis_3, Scoring_Method")
    print("Each hypothesis sheet contains parameter quality labels, numeric scores,")
    print("weighted contributions, Final Score, Rank, Parameters Scored, and coverage.")


if __name__ == "__main__":
    main()


Using input workbook: C:\Users\ASUS\Documents\PDB_Scoring_Work\Final_PDB_Data_Collection_ITK_scoring_ranges_corrected.xlsx
Scoring ranges loaded from worksheet: Scoring_Ranges
  Resolution (Å): high=(Decimal('1.35'), Decimal('1.97'), False), medium=(Decimal('1.97'), Decimal('2.58'), False), low=(Decimal('2.58'), Decimal('3.20'), True)
  Residue Count Difference: high=(Decimal('4'), Decimal('34'), False), medium=(Decimal('34'), Decimal('63'), False), low=(Decimal('63'), Decimal('93'), True)
  R-Value Difference: high=(Decimal('0.013'), Decimal('0.035'), False), medium=(Decimal('0.035'), Decimal('0.056'), False), low=(Decimal('0.056'), Decimal('0.078'), True)
  Clashscore: high=(Decimal('0'), Decimal('28'), False), medium=(Decimal('28'), Decimal('56'), False), low=(Decimal('56'), Decimal('84'), True)
  Ramachandran Outliers (%): high=(Decimal('0.0'), Decimal('5.0'), False), medium=(Decimal('5.0'), Decimal('9.9'), False), low=(Decimal('9.9'), Decimal('14.9'), True)
  Sidechain Outliers (%